# Nvidia "Other" Owner Estimation

This notebook further breaks down the "Other" category of Nvidia chip ownership
(pre-computed by `nvidia_owners` as Total − Hyperscalers − China) by subtracting
additional named owners:

**Remainder = Other (pre-computed) − Excluded Other Owners**

The list of excluded other owners (e.g. xAI, Tesla) is configurable via
`OTHER_OWNERS_TO_EXCLUDE` in the parameters cell below.

The pre-computed "Other" is read from CSVs exported by `nvidia_owners`, which
preserves the per-sample correlations from the Monte Carlo model. This avoids
the variance inflation that would occur from independently re-sampling each
owner's CI and subtracting them from the total.

In [31]:
import squigglepy as sq
import numpy as np
import pandas as pd
from squigglepy.numbers import K, M

sq.set_seed(42)
np.random.seed(42)
N_SAMPLES = 5000

# ==============================================
# ADJUSTABLE PARAMETER: which "other" owners to
# subtract before computing the remainder.
# Each name must match an Owner in the input CSVs
# (data_inputs/named_other_owners_totals.csv and
#  data_inputs/named_other_owners_by_chip.csv).
# ==============================================
OTHER_OWNERS_TO_EXCLUDE = ['xAI'] # ['xAI', 'Tesla', 'CoreWeave']

CHIP_TYPES = ['A100', 'A800', 'H100/H200', 'H800', 'H20', 'B200', 'B300']
US_CHIPS = ['A100', 'H100/H200', 'B200', 'B300']
CHINA_CHIPS = ['H20', 'H800', 'A800']
HYPERSCALERS = ['Microsoft', 'Meta', 'Amazon', 'Google', 'Oracle']
CHINA_OWNER = 'China'
H100_TOPS = 1979

CHIP_SPECS = {
    'A100':      {'tops': 624,  'tdp': 400},
    'A800':      {'tops': 624,  'tdp': 400},
    'H100/H200': {'tops': 1979, 'tdp': 700},
    'H800':      {'tops': 1979, 'tdp': 700},
    'H20':       {'tops': 296,  'tdp': 400},
    'B200':      {'tops': 5000, 'tdp': 1200},
    'B300':      {'tops': 5000, 'tdp': 1400},
}

## Load CSV exports

Read the pre-computed "Other" (= Total − Hyperscalers − China) from `nvidia_owners`,
plus hyperscaler/China data for the summary table. Reconstruct sample arrays by
fitting lognormal distributions to the 5th/95th percentiles via `sq.to()`.

In [32]:
def sample_from_ci(p5, p50, p95, n=N_SAMPLES):
    """Sample from a lognormal fitted to the 5th/95th percentile CI."""
    if p5 <= 0 or p95 <= 0:
        return np.zeros(n)
    return np.array(sq.to(float(p5), float(p95)) @ n)

def end_date_to_quarter(end_date_str):
    """Convert an end date like '6/30/2024' to 'Q2 2024'."""
    dt = pd.to_datetime(end_date_str)
    q = (dt.month - 1) // 3 + 1
    return f'Q{q} {dt.year}'

def quarter_date_strings(cq):
    """Return (start_date, end_date) as M/D/YYYY for a calendar quarter string like 'Q2 2024'."""
    q_num = int(cq[1])
    year = int(cq.split()[-1])
    month_starts = {1: 1, 2: 4, 3: 7, 4: 10}
    month_ends = {1: (3, 31), 2: (6, 30), 3: (9, 30), 4: (12, 31)}
    start = f"{month_starts[q_num]}/1/{year}"
    end_m, end_d = month_ends[q_num]
    end = f"{end_m}/{end_d}/{year}"
    return start, end

# Map CSV chip names to internal names
CHIP_NAME_MAP = {'H100': 'H100/H200'}

In [33]:
# ==============================================
# LOAD PRE-COMPUTED "OTHER" FROM NVIDIA_OWNERS
# ==============================================
# "Other" = Total Nvidia - Hyperscalers - China, computed with correlated samples
# in nvidia_owners and exported to separate CSVs.

other_base_by_chip_df = pd.read_csv('data_inputs/nvidia_other_cumulative_by_chip.csv')
other_base_by_chip_df['quarter'] = other_base_by_chip_df['End date'].apply(end_date_to_quarter)
other_base_by_chip_df['chip'] = other_base_by_chip_df['Chip type'].map(lambda x: CHIP_NAME_MAP.get(x, x))

other_base_totals_df = pd.read_csv('data_inputs/nvidia_other_cumulative_totals.csv')
other_base_totals_df['quarter'] = other_base_totals_df['End date'].apply(end_date_to_quarter)

CALENDAR_QUARTERS = list(dict.fromkeys(other_base_by_chip_df['quarter']))
print(f"Calendar quarters: {CALENDAR_QUARTERS[0]} to {CALENDAR_QUARTERS[-1]} ({len(CALENDAR_QUARTERS)} quarters)")

# Per-chip unit samples for Other: {quarter: {chip: np.array(N_SAMPLES)}}
other_base_running = {cq: {chip: np.zeros(N_SAMPLES) for chip in CHIP_TYPES} for cq in CALENDAR_QUARTERS}

for _, row in other_base_by_chip_df.iterrows():
    chip = row['chip']
    if chip not in CHIP_TYPES:
        continue
    cq = row['quarter']
    if cq not in CALENDAR_QUARTERS:
        continue
    other_base_running[cq][chip] = sample_from_ci(
        row['Number of Units (5th percentile)'],
        row['Number of Units'],
        row['Number of Units (95th percentile)'],
    )

# Total H100e samples for Other (from the totals CSV)
other_base_h100e = {cq: np.zeros(N_SAMPLES) for cq in CALENDAR_QUARTERS}

for _, row in other_base_totals_df.iterrows():
    cq = row['quarter']
    if cq not in CALENDAR_QUARTERS:
        continue
    other_base_h100e[cq] = sample_from_ci(
        row['H100e (5th percentile)'],
        row['Compute estimate in H100e (median)'],
        row['H100e (95th percentile)'],
    )

last_cq = CALENDAR_QUARTERS[-1]
print(f"\nOther (pre-computed) through {last_cq}:")
print(f"  H100e (median): {int(np.median(other_base_h100e[last_cq])):,}")
print(f"  Units (median): {int(sum(np.median(other_base_running[last_cq][c]) for c in CHIP_TYPES)):,}")

Calendar quarters: Q1 2022 to Q1 2026 (17 quarters)

Other (pre-computed) through Q1 2026:
  H100e (median): 4,724,530
  Units (median): 2,987,843


In [34]:
# ==============================================
# LOAD HYPERSCALER + CHINA + TOTAL DATA (for summary table only)
# ==============================================
# These are used in the summary table to show the full ownership breakdown.
# The actual remainder computation uses the pre-computed Other above.

# Total Nvidia H100e (for summary table)
nvidia_totals_df = pd.read_csv('csv_export/nvidia_cumulative_totals.csv')
nvidia_totals_df['quarter'] = nvidia_totals_df['End date'].apply(end_date_to_quarter)

total_h100e = {}
for _, row in nvidia_totals_df.iterrows():
    cq = row['quarter']
    total_h100e[cq] = sample_from_ci(
        row['Compute estimate in H100e (5th percentile)'],
        row['Compute estimate in H100e (median)'],
        row['Compute estimate in H100e (95th percentile)'],
    )

# Hyperscaler + China H100e (for summary table)
owners_totals_df = pd.read_csv('owners_csv_export/nvidia_owners_cumulative_totals.csv')
owners_totals_df['quarter'] = owners_totals_df['End date'].apply(end_date_to_quarter)

KNOWN_OWNERS = HYPERSCALERS + [CHINA_OWNER]
owner_h100e = {}
for owner in KNOWN_OWNERS:
    owner_h100e[owner] = {cq: np.zeros(N_SAMPLES) for cq in CALENDAR_QUARTERS}

for _, row in owners_totals_df.iterrows():
    owner = row['Owner']
    if owner not in KNOWN_OWNERS:
        continue
    cq = row['quarter']
    if cq not in CALENDAR_QUARTERS:
        continue
    owner_h100e[owner][cq] = sample_from_ci(
        row['H100e (5th percentile)'],
        row['Compute estimate in H100e (median)'],
        row['H100e (95th percentile)'],
    )

# Quick check
for owner in KNOWN_OWNERS:
    h100e = int(np.median(owner_h100e[owner][last_cq]))
    print(f"{owner} through {last_cq}: {h100e:,} H100e")

Microsoft through Q1 2026: 3,335,620 H100e
Meta through Q1 2026: 1,949,749 H100e
Amazon through Q1 2026: 1,491,277 H100e
Google through Q1 2026: 1,294,431 H100e
Oracle through Q1 2026: 1,092,311 H100e
China through Q1 2026: 400,664 H100e


## Other owner estimates

For each owner in `OTHER_OWNERS_TO_EXCLUDE`, two independent paths are built:

- **Cumulative H100e** (subtracted from the totals output) comes from `named_other_owners_totals.csv`.
- **Per-chip cumulative units** (subtracted from the by-chip output) come from `named_other_owners_by_chip.csv`.

When only one side is provided, the other is derived: totals-only owners are split into chip types using the overall non-China chip mix; chips-only owners get cumulative H100e summed from their per-chip units. If both are provided, each path uses its own input — they are not required to be consistent.

In [35]:
# ==============================================
# LOAD OTHER OWNER DATA FROM DATA_INPUTS
# ==============================================

other_totals_df = pd.read_csv('data_inputs/named_other_owners_totals.csv')
other_by_chip_df = pd.read_csv('data_inputs/named_other_owners_by_chip.csv')

# Parse end dates to calendar quarters
other_totals_df['end_dt'] = pd.to_datetime(other_totals_df['End date'], format='mixed')
other_totals_df['quarter'] = other_totals_df['end_dt'].apply(
    lambda dt: f'Q{(dt.month - 1) // 3 + 1} {dt.year}'
)

other_by_chip_df['end_dt'] = pd.to_datetime(other_by_chip_df['End date'], format='mixed')
other_by_chip_df['quarter'] = other_by_chip_df['end_dt'].apply(
    lambda dt: f'Q{(dt.month - 1) // 3 + 1} {dt.year}'
)
other_by_chip_df['chip'] = other_by_chip_df['Chip type'].map(lambda x: CHIP_NAME_MAP.get(x, x))

# Only include owners that appear in the CSV AND are in our exclude list
available_owners = set(other_totals_df['Owner'].unique()) | set(other_by_chip_df['Owner'].unique())
OTHER_OWNERS = [o for o in OTHER_OWNERS_TO_EXCLUDE if o in available_owners]

missing = set(OTHER_OWNERS_TO_EXCLUDE) - available_owners
if missing:
    print(f"WARNING: owners not found in input CSVs: {missing}")

print(f"Other owners to exclude: {OTHER_OWNERS}")
print(f"\nTotals data:\n{other_totals_df[other_totals_df['Owner'].isin(OTHER_OWNERS)][['Owner', 'quarter', 'Compute estimate in H100e (median)']].to_string()}")
print(f"\nBy-chip data:\n{other_by_chip_df[other_by_chip_df['Owner'].isin(OTHER_OWNERS)][['Owner', 'chip', 'quarter', 'Number of Units']].to_string()}")

Other owners to exclude: ['xAI']

Totals data:
  Owner  quarter  Compute estimate in H100e (median)
0   xAI  Q4 2024                              100000
1   xAI  Q4 2025                              553714

By-chip data:
  Owner       chip  quarter  Number of Units
0   xAI       B200  Q3 2025           140000
1   xAI       B200  Q4 2025           140000
2   xAI  H100/H200  Q3 2024           100000
3   xAI  H100/H200  Q4 2024           100000
4   xAI  H100/H200  Q1 2025           200000
5   xAI  H100/H200  Q3 2025           200000
6   xAI  H100/H200  Q4 2025           200000


In [36]:
# ==============================================
# INTERPOLATE TESLA H100e TO ALL QUARTERS, THEN SPLIT BY CHIP
# ==============================================
# Tesla only has total H100e (no chip breakdown), so we:
# 1. Interpolate cumulative H100e to every quarter
# 2. Split into chip types using the non-China chip mix from the total Nvidia model

DEFAULT_UNCERTAINTY = 0

def interpolate_cumulative_h100e(owner_rows, calendar_quarters):
    """Linearly interpolate cumulative H100e for one owner across all quarters.
    
    Returns {quarter: np.array(N_SAMPLES)} sampled with uncertainty.
    """
    cq_to_idx = {cq: i for i, cq in enumerate(calendar_quarters)}
    
    # Collect known data points
    data_points = []
    for _, row in owner_rows.iterrows():
        cq = row['quarter']
        if cq not in cq_to_idx:
            continue
        median = float(row['Compute estimate in H100e (median)'])
        p5 = row.get('H100e (5th percentile)')
        p95 = row.get('H100e (95th percentile)')
        p5 = None if pd.isna(p5) or str(p5).strip() == '' else float(p5)
        p95 = None if pd.isna(p95) or str(p95).strip() == '' else float(p95)
        data_points.append({'idx': cq_to_idx[cq], 'median': median, 'p5': p5, 'p95': p95})
    data_points.sort(key=lambda x: x['idx'])
    
    if not data_points:
        return {cq: np.zeros(N_SAMPLES) for cq in calendar_quarters}
    
    # Interpolate medians
    interpolated = np.zeros(len(calendar_quarters))
    first_idx, last_idx = data_points[0]['idx'], data_points[-1]['idx']
    
    for i in range(len(calendar_quarters)):
        if i < first_idx:
            interpolated[i] = 0.0
        elif i > last_idx:
            interpolated[i] = data_points[-1]['median']
        else:
            prev = next((p for p in reversed(data_points) if p['idx'] <= i), None)
            nxt = next((p for p in data_points if p['idx'] >= i), None)
            if prev['idx'] == nxt['idx']:
                interpolated[i] = prev['median']
            else:
                frac = (i - prev['idx']) / (nxt['idx'] - prev['idx'])
                interpolated[i] = prev['median'] + frac * (nxt['median'] - prev['median'])
    
    # Sample with uncertainty
    result = {}
    for i, cq in enumerate(calendar_quarters):
        med = interpolated[i]
        if med <= 0:
            result[cq] = np.zeros(N_SAMPLES)
            continue
        
        matching = next((p for p in data_points if p['idx'] == i), None)
        if matching and matching['p5'] is not None and matching['p95'] is not None:
            low, high = matching['p5'], matching['p95']
        else:
            low, high = med * (1 - DEFAULT_UNCERTAINTY), med * (1 + DEFAULT_UNCERTAINTY)
        
        result[cq] = np.array(sq.to(max(low, 1), high) @ N_SAMPLES)
    return result


def compute_us_chip_h100e_mix(chip_running, calendar_quarters):
    """Fraction of cumulative non-China H100e from each US chip type per quarter.
    
    Returns {quarter: {chip: np.array of fractions}}.
    """
    mix = {}
    for cq in calendar_quarters:
        chip_h100e = {}
        for chip in US_CHIPS:
            if chip in CHIP_SPECS:
                chip_h100e[chip] = chip_running[cq][chip] * (CHIP_SPECS[chip]['tops'] / H100_TOPS)
        total_h100e = sum(chip_h100e.values())
        
        mix[cq] = {}
        for chip in US_CHIPS:
            if chip in CHIP_SPECS:
                safe_total = np.where(total_h100e > 0, total_h100e, 1.0)
                mix[cq][chip] = np.where(total_h100e > 0, chip_h100e[chip] / safe_total, 0.0)
    return mix

h100e_chip_mix = compute_us_chip_h100e_mix(other_base_running, CALENDAR_QUARTERS)

In [37]:
# ==============================================
# BUILD PER-CHIP UNITS AND CUMULATIVE H100E FOR EACH OTHER OWNER
# ==============================================
# Two independent paths:
#   - Cumulative H100e comes from the totals CSV (drives totals-output subtraction)
#   - Per-chip units come from the by-chip CSV (drives by-chip-output subtraction)
# When one side is missing, it's derived from the other.

def interpolate_chip_units(chip_rows, calendar_quarters):
    """Linearly interpolate cumulative units for one chip across all quarters."""
    cq_to_idx = {cq: i for i, cq in enumerate(calendar_quarters)}
    data_points = []
    for _, row in chip_rows.iterrows():
        cq = row['quarter']
        if cq not in cq_to_idx:
            continue
        units = float(row['Number of Units'])
        p5 = row.get('Number of Units (5th percentile)')
        p95 = row.get('Number of Units (95th percentile)')
        p5 = None if pd.isna(p5) or str(p5).strip() == '' else float(p5)
        p95 = None if pd.isna(p95) or str(p95).strip() == '' else float(p95)
        data_points.append({'idx': cq_to_idx[cq], 'median': units, 'p5': p5, 'p95': p95})
    data_points.sort(key=lambda x: x['idx'])

    result = {cq: np.zeros(N_SAMPLES) for cq in calendar_quarters}
    if not data_points:
        return result

    interpolated = np.zeros(len(calendar_quarters))
    first_idx, last_idx = data_points[0]['idx'], data_points[-1]['idx']
    for i in range(len(calendar_quarters)):
        if i < first_idx:
            interpolated[i] = 0.0
        elif i > last_idx:
            interpolated[i] = data_points[-1]['median']
        else:
            prev = next((p for p in reversed(data_points) if p['idx'] <= i), None)
            nxt = next((p for p in data_points if p['idx'] >= i), None)
            if prev['idx'] == nxt['idx']:
                interpolated[i] = prev['median']
            else:
                frac = (i - prev['idx']) / (nxt['idx'] - prev['idx'])
                interpolated[i] = prev['median'] + frac * (nxt['median'] - prev['median'])

    for i, cq in enumerate(calendar_quarters):
        med = interpolated[i]
        if med <= 0:
            continue
        matching = next((p for p in data_points if p['idx'] == i), None)
        if matching and matching['p5'] is not None and matching['p95'] is not None:
            low, high = matching['p5'], matching['p95']
        else:
            low, high = med * (1 - DEFAULT_UNCERTAINTY), med * (1 + DEFAULT_UNCERTAINTY)
        result[cq] = np.array(sq.to(max(low, 1), high) @ N_SAMPLES)
    return result


other_owner_running = {}
other_owner_cumulative_h100e = {}

for owner in OTHER_OWNERS:
    owner_chip_data = other_by_chip_df[other_by_chip_df['Owner'] == owner]
    owner_total_data = other_totals_df[other_totals_df['Owner'] == owner]
    has_chip_breakdown = len(owner_chip_data) > 0
    has_totals = len(owner_total_data) > 0

    # ---- Per-chip units path (feeds by-chip output subtraction) ----
    other_owner_running[owner] = {cq: {chip: np.zeros(N_SAMPLES) for chip in CHIP_TYPES} for cq in CALENDAR_QUARTERS}

    if has_chip_breakdown:
        for chip in owner_chip_data['chip'].unique():
            if chip not in CHIP_TYPES:
                continue
            chip_rows = owner_chip_data[owner_chip_data['chip'] == chip].sort_values('end_dt')
            chip_units = interpolate_chip_units(chip_rows, CALENDAR_QUARTERS)
            for cq in CALENDAR_QUARTERS:
                other_owner_running[owner][cq][chip] = chip_units[cq]
    elif has_totals:
        # Fallback: split cumulative H100e by the non-China chip mix
        owner_h100e_interp = interpolate_cumulative_h100e(owner_total_data, CALENDAR_QUARTERS)
        for cq in CALENDAR_QUARTERS:
            for chip in US_CHIPS:
                if chip in CHIP_SPECS:
                    chip_h100e = owner_h100e_interp[cq] * h100e_chip_mix[cq][chip]
                    h100e_mult = CHIP_SPECS[chip]['tops'] / H100_TOPS
                    other_owner_running[owner][cq][chip] = np.where(h100e_mult > 0, chip_h100e / h100e_mult, 0.0)

    # ---- Cumulative H100e path (feeds totals output subtraction) ----
    if has_totals:
        other_owner_cumulative_h100e[owner] = interpolate_cumulative_h100e(owner_total_data, CALENDAR_QUARTERS)
    elif has_chip_breakdown:
        # Fallback: sum per-chip H100e
        other_owner_cumulative_h100e[owner] = {}
        for cq in CALENDAR_QUARTERS:
            h100e = np.zeros(N_SAMPLES)
            for chip in CHIP_TYPES:
                if chip in CHIP_SPECS:
                    h100e += other_owner_running[owner][cq][chip] * (CHIP_SPECS[chip]['tops'] / H100_TOPS)
            other_owner_cumulative_h100e[owner][cq] = h100e
    else:
        other_owner_cumulative_h100e[owner] = {cq: np.zeros(N_SAMPLES) for cq in CALENDAR_QUARTERS}

# Summary
for owner in OTHER_OWNERS:
    h100e_med = int(np.median(other_owner_cumulative_h100e[owner][last_cq]))
    units_med = int(sum(np.median(other_owner_running[owner][last_cq][c]) for c in CHIP_TYPES))
    print(f"{owner} through {last_cq}: {h100e_med:,} H100e, {units_med:,} units")

xAI through Q1 2026: 553,714 H100e, 340,000 units


## Compute remainder

Remainder = Pre-computed Other − Excluded Other Owners, computed per-chip to preserve chip-level detail.

In [38]:
# ==============================================
# REMAINDER = PRE-COMPUTED OTHER - EXCLUDED OWNERS (e.g. xAI)
# ==============================================
# The pre-computed Other already has the hyperscaler and China subtraction done
# with proper correlations. We only subtract the additional excluded owners here.

# Per-chip unit remainder
remainder_running = {}
for cq in CALENDAR_QUARTERS:
    remainder_running[cq] = {}
    for chip in CHIP_TYPES:
        chip_others = sum(other_owner_running[owner][cq][chip] for owner in OTHER_OWNERS)
        remainder_running[cq][chip] = other_base_running[cq][chip] - chip_others

# H100e remainder
remainder_h100e = {}
for cq in CALENDAR_QUARTERS:
    other_h100e_sum = sum(other_owner_cumulative_h100e[owner][cq] for owner in OTHER_OWNERS)
    remainder_h100e[cq] = other_base_h100e[cq] - other_h100e_sum

# Summary
all_owner_labels = HYPERSCALERS + [CHINA_OWNER] + OTHER_OWNERS + ['Remainder']

print(f"Cumulative H100e through {last_cq}:")
print(f"  {'Total Nvidia':<20}{int(np.median(total_h100e[last_cq])):>12,}")
for owner in KNOWN_OWNERS:
    print(f"  {owner:<20}{int(np.median(owner_h100e[owner][last_cq])):>12,}")
print(f"  {'Other (base)':<20}{int(np.median(other_base_h100e[last_cq])):>12,}")
for owner in OTHER_OWNERS:
    print(f"  {owner:<20}{int(np.median(other_owner_cumulative_h100e[owner][last_cq])):>12,}")
print(f"  {'Remainder':<20}{int(np.median(remainder_h100e[last_cq])):>12,}")

Cumulative H100e through Q1 2026:
  Total Nvidia          14,450,971
  Microsoft              3,335,620
  Meta                   1,949,749
  Amazon                 1,491,277
  Google                 1,294,431
  Oracle                 1,092,311
  China                    400,664
  Other (base)           4,724,530
  xAI                      553,714
  Remainder              4,170,816


In [39]:
# ==============================================
# FULL SUMMARY TABLE: H100e BY OWNER OVER TIME
# ==============================================

owner_h100e_series = {}
for owner in KNOWN_OWNERS:
    owner_h100e_series[owner] = owner_h100e[owner]
for owner in OTHER_OWNERS:
    owner_h100e_series[owner] = other_owner_cumulative_h100e[owner]
owner_h100e_series['Remainder'] = remainder_h100e

rows = []
for cq in CALENDAR_QUARTERS:
    row = {'Quarter': cq}
    row['Total'] = int(np.median(total_h100e[cq]))
    for label in all_owner_labels:
        row[label] = int(np.median(owner_h100e_series[label][cq]))
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index('Quarter')
print("Cumulative H100e by owner (median):\n")
print(summary_df.to_string())

Cumulative H100e by owner (median):

            Total  Microsoft     Meta   Amazon   Google   Oracle   China     xAI  Remainder
Quarter                                                                                    
Q1 2022     66826      14775     7758     5707     5665     1616    9441       0      17812
Q2 2022    138994      30626    16117    11881    11599     3317   19608       0      36718
Q3 2022    213256      47313    24894    18495    18040     5194   22818       0      65544
Q4 2022    289904      64993    34246    25055    24604     7797   32424       0      90033
Q1 2023    383144      86541    44774    32914    32436    11283   49235       0     115048
Q2 2023    594648     133002    69504    50847    50618    19127   89205       0     169098
Q3 2023    933576     208890   108649    79116    79150    32085  154241       0     257101
Q4 2023   1382235     308462   161789   116583   117280    54652  177722       0     432011
Q1 2024   1958383     441680   233052   168

## Export CSVs

Export the "Other" remainder to `owners_csv_export/` in the same format as the original notebook.

In [40]:
# ==============================================
# EXPORT TO CSV
# ==============================================

from datetime import datetime as _dt

timestamp = _dt.now().strftime("%m-%d-%Y %H:%M")
output_dir = 'owners_csv_export'

# Determine incomplete quarters from the Other base totals CSV
incomplete_quarters = set()
for _, row in other_base_totals_df.iterrows():
    if str(row.get('Incomplete', '')).strip().lower() == 'checked':
        incomplete_quarters.add(row['quarter'])

named_owners = HYPERSCALERS + [CHINA_OWNER] + OTHER_OWNERS
named_owners_str = ', '.join(named_owners)

XAI_TIMING_NOTE = "Other may be artificially low due to xAI timing."

def tail_cols(cq=None, extra_note=None):
    base_note = f"Excludes {named_owners_str}."
    if cq in incomplete_quarters:
        base_note += " Incomplete: based on Nvidia fiscal quarters ending 1/25/2026."
    if extra_note:
        base_note += f" {extra_note}"
    base_note += f" Estimates generated on: {timestamp}"
    return {
        'Source / Link': '',
        'Notes': base_note,
    }

OWNER_LABEL = 'Other'
# Only export non-China chip types (China chips are allocated entirely to the China owner)
EXPORT_CHIPS = [c for c in CHIP_TYPES if c not in CHINA_CHIPS]
first_q_start, _ = quarter_date_strings(CALENDAR_QUARTERS[0])

# --- 1. Cumulative totals CSV ---
# Use remainder_h100e (sampled from authoritative CSV totals) for H100e columns,
# and per-chip unit remainder summed for the units columns.
cumulative_rows = []
for cq in CALENDAR_QUARTERS:
    unit_samples = sum(remainder_running[cq][chip] for chip in EXPORT_CHIPS)
    h100e_samples = remainder_h100e[cq]
    power_w = sum(
        remainder_running[cq][chip] * CHIP_SPECS[chip]['tdp']
        for chip in EXPORT_CHIPS if chip in CHIP_SPECS
    )
    power_mw = power_w / 1e6

    _, end_date = quarter_date_strings(cq)
    row = {
        'Name': f"Other cumulative Nvidia through {cq}",
        'Chip manufacturer': 'Nvidia',
        'Owner': OWNER_LABEL,
        'Start date': first_q_start,
        'End date': end_date,
        'Compute estimate in H100e (median)': int(np.percentile(h100e_samples, 50)),
        'H100e (5th percentile)': int(np.percentile(h100e_samples, 5)),
        'H100e (95th percentile)': int(np.percentile(h100e_samples, 95)),
        'Number of Units': int(np.percentile(unit_samples, 50)),
        'Number of Units (5th percentile)': int(np.percentile(unit_samples, 5)),
        'Number of Units (95th percentile)': int(np.percentile(unit_samples, 95)),
        'Total TDP (W)': int(np.percentile(power_w, 50)),
        'Total TDP (W) (5th percentile)': int(np.percentile(power_w, 5)),
        'Total TDP (W) (95th percentile)': int(np.percentile(power_w, 95)),
        'Power in MW (median)': round(np.percentile(power_mw, 50), 2),
        'Power in MW (5th percentile)': round(np.percentile(power_mw, 5), 2),
        'Power in MW (95th percentile)': round(np.percentile(power_mw, 95), 2),
        'Incomplete': 'checked' if cq in incomplete_quarters else '',
    }
    row.update(tail_cols(cq))
    cumulative_rows.append(row)

cumulative_df = pd.DataFrame(cumulative_rows)
cumulative_df.to_csv(f'{output_dir}/nvidia_owners_OTHER_cumulative_totals.csv', index=False)
print(f"Exported {len(cumulative_df)} rows to {output_dir}/nvidia_owners_OTHER_cumulative_totals.csv")

# --- 2. Cumulative by chip type CSV ---
# Derive per-chip H100e from unit remainder, then scale so that chip H100e medians
# sum to the authoritative totals H100e median. This anchors the per-chip breakdown
# to the totals while preserving the relative chip mix from the unit model.
by_chip_rows = []
for cq in CALENDAR_QUARTERS:
    # First compute unscaled per-chip H100e from unit remainder
    chip_h100e_raw = {}
    for chip in EXPORT_CHIPS:
        if chip not in CHIP_SPECS:
            continue
        unit_samples = remainder_running[cq][chip]
        if np.median(unit_samples) <= 0:
            continue
        chip_h100e_raw[chip] = unit_samples * (CHIP_SPECS[chip]['tops'] / H100_TOPS)

    # Scale factor: ratio of totals H100e median to sum-of-chips H100e median
    if chip_h100e_raw:
        raw_sum_median = sum(np.median(v) for v in chip_h100e_raw.values())
        total_median = np.median(remainder_h100e[cq])
        scale = total_median / raw_sum_median if raw_sum_median > 0 else 1.0
    else:
        scale = 1.0

    for chip in EXPORT_CHIPS:
        if chip not in chip_h100e_raw:
            continue
        unit_samples = remainder_running[cq][chip]
        h100e_samples = chip_h100e_raw[chip] * scale
        tdp_w = unit_samples * CHIP_SPECS[chip]['tdp']

        _, end_date = quarter_date_strings(cq)
        row = {
            'Name': f"Other {chip} cumulative through {cq}",
            'Chip manufacturer': 'Nvidia',
            'Owner': OWNER_LABEL,
            'Start date': first_q_start,
            'End date': end_date,
            'Compute estimate in H100e (median)': int(np.percentile(h100e_samples, 50)),
            'H100e (5th percentile)': int(np.percentile(h100e_samples, 5)),
            'H100e (95th percentile)': int(np.percentile(h100e_samples, 95)),
            'Number of Units': int(np.percentile(unit_samples, 50)),
            'Number of Units (5th percentile)': int(np.percentile(unit_samples, 5)),
            'Number of Units (95th percentile)': int(np.percentile(unit_samples, 95)),
            'Total TDP (W)': int(np.percentile(tdp_w, 50)),
            'Total TDP (W) (5th percentile)': int(np.percentile(tdp_w, 5)),
            'Total TDP (W) (95th percentile)': int(np.percentile(tdp_w, 95)),
            'Chip type': chip,
            'Incomplete': 'checked' if cq in incomplete_quarters else '',
        }
        row.update(tail_cols(cq))
        by_chip_rows.append(row)

by_chip_df = pd.DataFrame(by_chip_rows)
by_chip_df.to_csv(f'{output_dir}/nvidia_owners_OTHER_cumulative_by_chip.csv', index=False)
print(f"Exported {len(by_chip_df)} rows to {output_dir}/nvidia_owners_OTHER_cumulative_by_chip.csv")

# Sanity check: compare by-chip H100e sum vs totals H100e for last quarter
last_end = quarter_date_strings(CALENDAR_QUARTERS[-1])[1]
last_chips = by_chip_df[by_chip_df['End date'] == last_end]
chip_h100e_sum = last_chips['Compute estimate in H100e (median)'].sum()
total_h100e_med = cumulative_df.iloc[-1]['Compute estimate in H100e (median)']
print(f"\n{CALENDAR_QUARTERS[-1]} sanity check:")
print(f"  By-chip H100e sum: {int(chip_h100e_sum):,}")
print(f"  Totals H100e:      {int(total_h100e_med):,}")
print(f"  Difference:        {int(chip_h100e_sum - total_h100e_med):,}")

# --- 3. Quarterly (non-cumulative) by chip type CSV ---
# Use pre-computed Other quarterly from nvidia_owners (preserves correlations)
other_base_q_df = pd.read_csv('data_inputs/nvidia_other_quarters_by_chip.csv')
other_base_q_df['quarter'] = other_base_q_df['End date'].apply(end_date_to_quarter)
other_base_q_df['chip'] = other_base_q_df['Chip type'].map(lambda x: CHIP_NAME_MAP.get(x, x))

# Sample Other quarterly per chip
other_base_quarterly = {cq: {chip: np.zeros(N_SAMPLES) for chip in CHIP_TYPES} for cq in CALENDAR_QUARTERS}
for _, row in other_base_q_df.iterrows():
    chip = row['chip']
    cq = row['quarter']
    if chip not in CHIP_TYPES or cq not in CALENDAR_QUARTERS:
        continue
    other_base_quarterly[cq][chip] = sample_from_ci(
        row['Number of Units (5th percentile)'],
        row['Number of Units'],
        row['Number of Units (95th percentile)'],
    )

# Compute per-quarter other owner units (diff consecutive cumulative)
other_owner_quarterly = {}
for owner in OTHER_OWNERS:
    for i, cq in enumerate(CALENDAR_QUARTERS):
        for chip in CHIP_TYPES:
            now = other_owner_running[owner][cq][chip]
            prev = other_owner_running[owner][CALENDAR_QUARTERS[i - 1]][chip] if i > 0 else np.zeros(N_SAMPLES)
            q_units = now - prev
            if np.median(q_units) <= 0:
                continue
            other_owner_quarterly.setdefault(cq, {}).setdefault(chip, np.zeros(N_SAMPLES))
            other_owner_quarterly[cq][chip] += q_units

# Remainder quarterly = Other quarterly - excluded owners quarterly
quarterly_by_chip_rows = []
for cq in CALENDAR_QUARTERS:
    start_date, end_date = quarter_date_strings(cq)
    for chip in EXPORT_CHIPS:
        if chip not in CHIP_SPECS:
            continue
        base_q = other_base_quarterly[cq].get(chip, np.zeros(N_SAMPLES))
        other_q = other_owner_quarterly.get(cq, {}).get(chip, np.zeros(N_SAMPLES))
        remainder_q = base_q - other_q

        # Skip chips with no base data this quarter
        if np.median(base_q) <= 0:
            continue

        # Clip negative remainders to 0 and flag with a note
        clipped = np.median(remainder_q) <= 0
        remainder_q = np.clip(remainder_q, 0, None)

        h100e_q = remainder_q * (CHIP_SPECS[chip]['tops'] / H100_TOPS)
        tdp_q = remainder_q * CHIP_SPECS[chip]['tdp']

        extra_note = XAI_TIMING_NOTE if clipped else None
        row = {
            'Name': f"Other {chip} {cq}",
            'Chip manufacturer': 'Nvidia',
            'Owner': OWNER_LABEL,
            'Start date': start_date,
            'End date': end_date,
            'Compute estimate in H100e (median)': int(np.percentile(h100e_q, 50)),
            'H100e (5th percentile)': int(np.percentile(h100e_q, 5)),
            'H100e (95th percentile)': int(np.percentile(h100e_q, 95)),
            'Number of Units': int(np.percentile(remainder_q, 50)),
            'Number of Units (5th percentile)': int(np.percentile(remainder_q, 5)),
            'Number of Units (95th percentile)': int(np.percentile(remainder_q, 95)),
            'Total TDP (W)': int(np.percentile(tdp_q, 50)),
            'Total TDP (W) (5th percentile)': int(np.percentile(tdp_q, 5)),
            'Total TDP (W) (95th percentile)': int(np.percentile(tdp_q, 95)),
            'Chip type': chip,
            'Incomplete': 'checked' if cq in incomplete_quarters else '',
        }
        row.update(tail_cols(cq, extra_note=extra_note))
        quarterly_by_chip_rows.append(row)

quarterly_by_chip_df = pd.DataFrame(quarterly_by_chip_rows)
quarterly_by_chip_df.to_csv(f'{output_dir}/nvidia_owners_OTHER_quarters_by_chip.csv', index=False)
print(f"Exported {len(quarterly_by_chip_df)} rows to {output_dir}/nvidia_owners_OTHER_quarters_by_chip.csv")

Exported 17 rows to owners_csv_export/nvidia_owners_OTHER_cumulative_totals.csv
Exported 42 rows to owners_csv_export/nvidia_owners_OTHER_cumulative_by_chip.csv

Q1 2026 sanity check:
  By-chip H100e sum: 4,170,813
  Totals H100e:      4,170,816
  Difference:        -3
Exported 32 rows to owners_csv_export/nvidia_owners_OTHER_quarters_by_chip.csv
